# Testes de Qualidade de Dados com pandera — `olist_orders_dataset`

Este notebook usa [pandera](https://pandera.readthedocs.io/en/stable/) para testar as principais dimensões de qualidade de dados no dataset `olist_orders_dataset.csv`:

1. **Completude** — campos obrigatórios não podem ter nulos; campos opcionais têm taxa de nulos dentro do esperado.
2. **Unicidade** — a chave primária (`order_id`) não pode se repetir.
3. **Validade** — formato dos IDs, domínio de `order_status` e formato dos timestamps.
4. **Consistência** — relações lógicas entre colunas (ex.: aprovação não pode ser anterior à compra).
5. **Atualidade / Faixa temporal** — as datas devem cair dentro da janela operacional conhecida do dataset.

Cada dimensão usa `schema.validate(df, lazy=True)` para acumular todas as falhas de uma vez, em vez de parar na primeira erro encontrado.

In [1]:
import numpy as np
import pandas as pd
import pandera.pandas as pa
from pandera.errors import SchemaErrors

pd.set_option("display.max_colwidth", 60)

RAW_PATH = "../data/raw/olist_orders_dataset.csv"

# Tudo como string, para testar o dado "cru" (nulos, formato, domínio)
df_raw = pd.read_csv(RAW_PATH, dtype=str)

DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# Cópia com as datas parseadas, usada nos testes de consistência e atualidade
df_dates = df_raw.copy()
for col in DATE_COLS:
    df_dates[col] = pd.to_datetime(df_dates[col], errors="coerce")

df_raw.shape

(99441, 8)

In [2]:
results = []


def run_check(dimension: str, schema: pa.DataFrameSchema, data: pd.DataFrame) -> None:
    """Valida `data` com `schema` (lazy=True) e registra o resultado em `results`."""
    try:
        schema.validate(data, lazy=True)
    except SchemaErrors as exc:
        failures = exc.failure_cases
        n_rows = failures["index"].nunique()
        results.append({"dimensao": dimension, "status": "FALHOU", "n_falhas": n_rows})
        print(f"[{dimension}] FALHOU — {n_rows} linha(s) com violação:")
        display(failures.head(10))
    else:
        results.append({"dimensao": dimension, "status": "OK", "n_falhas": 0})
        print(f"[{dimension}] OK — nenhuma violação encontrada.")

## 1. Completude

Colunas de identificação e a data de compra são obrigatórias (`nullable=False`). As colunas do ciclo de vida do pedido (`order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`) podem ter nulos legítimos (o pedido ainda não avançou nessa etapa), mas a taxa de nulos precisa ficar abaixo de um limite aceitável (5%).

In [3]:
def taxa_de_nulos_abaixo_de(limite: float) -> pa.Check:
    return pa.Check(
        lambda s: s.isna().mean() < limite,
        ignore_na=False,
        name=f"taxa_de_nulos_abaixo_de_{int(limite * 100)}pct",
    )


completeness_schema = pa.DataFrameSchema(
    {
        "order_id": pa.Column(str, nullable=False),
        "customer_id": pa.Column(str, nullable=False),
        "order_status": pa.Column(str, nullable=False),
        "order_purchase_timestamp": pa.Column(str, nullable=False),
        "order_estimated_delivery_date": pa.Column(str, nullable=False),
        "order_approved_at": pa.Column(str, nullable=True, checks=taxa_de_nulos_abaixo_de(0.05)),
        "order_delivered_carrier_date": pa.Column(
            str, nullable=True, checks=taxa_de_nulos_abaixo_de(0.05)
        ),
        "order_delivered_customer_date": pa.Column(
            str, nullable=True, checks=taxa_de_nulos_abaixo_de(0.05)
        ),
    },
    strict=False,
)

run_check("Completude", completeness_schema, df_raw)

[Completude] OK — nenhuma violação encontrada.


## 2. Unicidade

`order_id` é a chave primária da tabela de pedidos — cada pedido deve aparecer uma única vez.

In [4]:
uniqueness_schema = pa.DataFrameSchema(
    {
        "order_id": pa.Column(str, unique=True),
    },
    strict=False,
)

run_check("Unicidade", uniqueness_schema, df_raw)

[Unicidade] OK — nenhuma violação encontrada.


## 3. Validade

- `order_id` e `customer_id` devem ser hashes hexadecimais de 32 caracteres.
- `order_status` deve pertencer ao domínio conhecido de status.
- As colunas de timestamp devem seguir o formato `AAAA-MM-DD HH:MM:SS`.

In [5]:
ID_PATTERN = r"^[0-9a-f]{32}$"
TIMESTAMP_PATTERN = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
VALID_STATUSES = [
    "delivered",
    "shipped",
    "canceled",
    "unavailable",
    "invoiced",
    "processing",
    "created",
    "approved",
]

validity_schema = pa.DataFrameSchema(
    {
        "order_id": pa.Column(str, pa.Check.str_matches(ID_PATTERN)),
        "customer_id": pa.Column(str, pa.Check.str_matches(ID_PATTERN)),
        "order_status": pa.Column(str, pa.Check.isin(VALID_STATUSES)),
        "order_purchase_timestamp": pa.Column(str, pa.Check.str_matches(TIMESTAMP_PATTERN)),
        "order_approved_at": pa.Column(str, pa.Check.str_matches(TIMESTAMP_PATTERN), nullable=True),
        "order_delivered_carrier_date": pa.Column(
            str, pa.Check.str_matches(TIMESTAMP_PATTERN), nullable=True
        ),
        "order_delivered_customer_date": pa.Column(
            str, pa.Check.str_matches(TIMESTAMP_PATTERN), nullable=True
        ),
        "order_estimated_delivery_date": pa.Column(str, pa.Check.str_matches(TIMESTAMP_PATTERN)),
    },
    strict=False,
)

run_check("Validade", validity_schema, df_raw)

[Validade] OK — nenhuma violação encontrada.


## 4. Consistência

Relações lógicas que precisam valer entre colunas, verificadas com *checks* de `DataFrameSchema` (recebem o DataFrame inteiro):

- aprovação não pode ser anterior à compra;
- postagem para a transportadora não pode ser anterior à aprovação;
- entrega ao cliente não pode ser anterior à postagem;
- pedidos com `order_status == "delivered"` precisam ter `order_delivered_customer_date` preenchida.

In [6]:
def aprovacao_apos_compra(df: pd.DataFrame) -> pd.Series:
    mask = df["order_approved_at"].notna()
    ok = np.where(mask, df["order_approved_at"] >= df["order_purchase_timestamp"], True)
    return pd.Series(ok, index=df.index)


def postagem_apos_aprovacao(df: pd.DataFrame) -> pd.Series:
    mask = df["order_delivered_carrier_date"].notna() & df["order_approved_at"].notna()
    ok = np.where(mask, df["order_delivered_carrier_date"] >= df["order_approved_at"], True)
    return pd.Series(ok, index=df.index)


def entrega_apos_postagem(df: pd.DataFrame) -> pd.Series:
    mask = df["order_delivered_customer_date"].notna() & df["order_delivered_carrier_date"].notna()
    ok = np.where(
        mask, df["order_delivered_customer_date"] >= df["order_delivered_carrier_date"], True
    )
    return pd.Series(ok, index=df.index)


def status_entregue_tem_data_entrega(df: pd.DataFrame) -> pd.Series:
    mask = df["order_status"] == "delivered"
    ok = np.where(mask, df["order_delivered_customer_date"].notna(), True)
    return pd.Series(ok, index=df.index)


consistency_schema = pa.DataFrameSchema(
    columns={
        "order_status": pa.Column(str),
        "order_purchase_timestamp": pa.Column("datetime64[ns]"),
        "order_approved_at": pa.Column("datetime64[ns]", nullable=True),
        "order_delivered_carrier_date": pa.Column("datetime64[ns]", nullable=True),
        "order_delivered_customer_date": pa.Column("datetime64[ns]", nullable=True),
    },
    checks=[
        pa.Check(aprovacao_apos_compra, element_wise=False, name="aprovacao_apos_compra"),
        pa.Check(postagem_apos_aprovacao, element_wise=False, name="postagem_apos_aprovacao"),
        pa.Check(entrega_apos_postagem, element_wise=False, name="entrega_apos_postagem"),
        pa.Check(
            status_entregue_tem_data_entrega,
            element_wise=False,
            name="status_entregue_tem_data_entrega",
        ),
    ],
    strict=False,
)

run_check("Consistência", consistency_schema, df_dates)

[Consistência] FALHOU — 1390 linha(s) com violação:


,schema_context,column,check,check_number,failure_case,index
0,DataFrameSchema,order_id,postagem_apos_aprovacao,1,dcb36b511fcac050b97cd5c05de84dc3,15
7405,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-07-04 16:17:00,44573
7397,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-08-23 13:05:00,44167
7398,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-04-23 22:11:37,44238
7399,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-07-04 14:31:00,44272
7400,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-07-05 13:56:00,44295
7401,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-07-23 10:15:00,44399
7402,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-04-24 17:04:21,44425
7403,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2018-07-04 14:30:00,44452
7404,DataFrameSchema,order_delivered_carrier_date,postagem_apos_aprovacao,1,2017-08-01 15:54:30,44509


## 5. Atualidade / Faixa temporal

As datas devem cair dentro da janela operacional conhecida do dataset (2016 a 2018). Datas fora desse intervalo (ex.: `1970-01-01`, datas futuras) costumam indicar placeholders ou erros de carga — um problema clássico de *timeliness*.

In [7]:
WINDOW_START = pd.Timestamp("2016-01-01")
WINDOW_END = pd.Timestamp("2018-12-31")

timeliness_schema = pa.DataFrameSchema(
    {
        col: pa.Column(
            "datetime64[ns]",
            pa.Check.in_range(WINDOW_START, WINDOW_END),
            nullable=True,
        )
        for col in DATE_COLS
    },
    strict=False,
)

run_check("Atualidade", timeliness_schema, df_dates)

[Atualidade] OK — nenhuma violação encontrada.


## Resumo

In [8]:
pd.DataFrame(results)

,dimensao,status,n_falhas
0,Completude,OK,0
1,Unicidade,OK,0
2,Validade,OK,0
3,Consistência,FALHOU,1390
4,Atualidade,OK,0
